# Exercício 11 — AutoGen Research Lab

**Entrega sem ecrã:** código executável abaixo + documentação em `docs/`.


## 0. Ambiente e caminhos

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv

EX_ROOT = Path.cwd().resolve()
REPO_ROOT = EX_ROOT.parent.parent

load_dotenv(REPO_ROOT / ".env", override=False)
load_dotenv(EX_ROOT / ".env", override=True)

if not (os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")):
    raise RuntimeError("Defina GOOGLE_API_KEY no .env na raiz do repositório.")

print("OK — Repo:", REPO_ROOT)
print("OK — Exercício:", EX_ROOT)


## Rodadas Pesquisador ↔ Crítico → Sintetizador

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser
import os

llm = ChatGoogleGenerativeAI(
    model=(os.environ.get("GEMINI_MODEL") or "gemini-2.0-flash").replace("models/", ""),
    temperature=0.4,
)

def say(role: str, instr: str, **kw):
    p = ChatPromptTemplate.from_messages([
        ("system", role),
        ("human", instr),
    ])
    return (p | llm | StrOutputParser()).invoke(kw)

tema = "Impacto regulatório da IA generativa em PME europeias."
r1 = say("És um pesquisador.", "Proposta inicial sobre {tema}.", tema=tema)
r2 = say("És um crítico exigente.", "Critica:\n{r1}", r1=r1)
r3 = say("És o pesquisador.", "Ajusta com base na crítica:\n{r2}", r2=r2)
final = say("És sintetizador.", "Relatório final conciso:\nPesquisa:{r1}\nCrítica:{r2}\nReposta:{r3}", r1=r1, r2=r2, r3=r3)
print(final)


## Referências
- `docs/` desta pasta — explicações detalhadas.
- `app/` — espelho API opcional.
